In [ ]:
import os
import tarfile
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets.utils import download_url
from transformers import (
    CLIPProcessor,
    CLIPModel,
    CLIPTokenizer,
    GPT2Tokenizer,
    GPT2LMHeadModel
)
from tqdm import tqdm

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device}")

In [ ]:
# def prepare_dataset():
#     if os.path.exists("Images"):
#         print("Dataset already exists. Skipping download.")
#         return

#     print("Downloading Stanford Dogs Dataset (750MB)...")
#     url = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"
#     download_url(url, root=".", filename="images.tar")

#     print("Extracting images...")
#     with tarfile.open("images.tar") as tar:
#         tar.extractall(path=".")

#     print("Generating VQA Pairs...")
#     data = []
#     base_dir = "Images"

#     for breed_folder in os.listdir(base_dir):
#         folder_path = os.path.join(base_dir, breed_folder)
#         if os.path.isdir(folder_path):
#             breed_name = breed_folder.split("-")[-1].replace("_", " ").lower()

#             for img_file in os.listdir(folder_path):
#                 rel_path = os.path.join(breed_folder, img_file)
#                 data.append({
#                     'image_name': rel_path,
#                     'question': "What breed of dog is this?",
#                     'answer': breed_name
#                 })

#     df = pd.DataFrame(data)
#     df = df.sample(frac=1, random_state=42).reset_index(drop=True)
#     df.to_csv("dog_vqa_dataset.csv", index=False)
#     print(f"Created Dataset with {len(df)} images.")

# prepare_dataset()

In [ ]:
class ManualCrossAttention(nn.Module):
    def __init__(self, dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.ln = nn.LayerNorm(dim)

    def forward(self, text_embeds, image_embeds):
        B, T_q, C = text_embeds.shape
        B, T_kv, _ = image_embeds.shape

        q = self.q_proj(text_embeds).view(B, T_q, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(image_embeds).view(B, T_kv, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(image_embeds).view(B, T_kv, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) * self.scale
        attn_probs = attn_scores.softmax(dim=-1)

        context = (attn_probs @ v).transpose(1, 2).contiguous()
        context = context.view(B, T_q, C)
        output = self.out_proj(context)

        return self.ln(text_embeds + output)

In [ ]:
class SimpleVQAModel(nn.Module):
    def __init__(self, clip_model, decoder_model):
        super().__init__()
        self.clip = clip_model
        self.decoder = decoder_model

        vision_hidden = clip_model.config.vision_config.hidden_size
        clip_text_hidden = clip_model.config.text_config.hidden_size
        decoder_hidden = decoder_model.config.hidden_size

        self.image_proj = nn.Linear(vision_hidden, decoder_hidden)
        self.text_proj = nn.Linear(clip_text_hidden, decoder_hidden)
        self.cross_attn = ManualCrossAttention(decoder_hidden)

    def forward(self, pixel_values, clip_input_ids, gpt_input_ids, labels=None):
        with torch.no_grad():
            image_feats = self.clip.vision_model(pixel_values=pixel_values).last_hidden_state
            text_feats  = self.clip.text_model(input_ids=clip_input_ids).last_hidden_state

        image_feats = self.image_proj(image_feats)
        text_feats  = self.text_proj(text_feats)
        fused_feats = self.cross_attn(text_feats, image_feats)

        question_embeds = self.decoder.transformer.wte(gpt_input_ids)
        inputs_embeds = torch.cat([fused_feats, question_embeds], dim=1)

        aligned_labels = None
        if labels is not None:
            prefix_len = inputs_embeds.size(1) - labels.size(1)
            pad_labels = torch.full((labels.size(0), prefix_len), -100, device=labels.device)
            aligned_labels = torch.cat([pad_labels, labels], dim=1)

        outputs = self.decoder(inputs_embeds=inputs_embeds, labels=aligned_labels)
        return outputs

In [ ]:
class VQADataset(Dataset):
    def __init__(self, csv_path, image_folder, clip_processor, clip_tokenizer, gpt_tokenizer, max_length=32):
        self.df = pd.read_csv(csv_path)
        self.image_folder = image_folder
        self.clip_processor = clip_processor
        self.clip_tokenizer = clip_tokenizer
        self.gpt_tokenizer = gpt_tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_folder, row["image_name"])

        image = Image.open(img_path).convert("RGB")
        pixel_values = self.clip_processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)

        question = str(row["question"])
        answer = str(row["answer"])

        clip_q = self.clip_tokenizer(
            question, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt"
        )

        gpt_q = self.gpt_tokenizer(
            question, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt"
        )

        gpt_a = self.gpt_tokenizer(
            answer, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt"
        )

        return {
            "pixel_values": pixel_values,
            "clip_input_ids": clip_q["input_ids"].squeeze(0),
            "gpt_input_ids": gpt_q["input_ids"].squeeze(0),
            "labels": gpt_a["input_ids"].squeeze(0)
        }

In [ ]:

csv_path = r"E:/@IIT_BBS/@Sem 1/AI Lab/Project-GeoVQA/ENCODER/Dataset/vqa_filtered.csv"
image_folder = "E:/@IIT_BBS/@Sem 1/AI Lab/Project-GeoVQA/ENCODER/Dataset/images/train2014"

In [ ]:
print("Loading Pretrained Models...")
clip_id = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_id)
clip_tokenizer = CLIPTokenizer.from_pretrained(clip_id)
clip_model = CLIPModel.from_pretrained(clip_id)

gpt_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token
decoder_model = GPT2LMHeadModel.from_pretrained("gpt2")
print("Loading done!")

In [ ]:
vqa_model = SimpleVQAModel(clip_model, decoder_model).to(device)

for p in vqa_model.clip.parameters(): p.requires_grad = False
for p in vqa_model.decoder.parameters(): p.requires_grad = False

dataset = VQADataset(
    csv_path=csv_path,
    image_folder=image_folder,
    clip_processor = clip_processor,
    clip_tokenizer = clip_tokenizer,
    gpt_tokenizer= gpt_tokenizer
    )

train_loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

optimizer = torch.optim.AdamW(
    list(vqa_model.image_proj.parameters()) +
    list(vqa_model.text_proj.parameters()) +
    list(vqa_model.cross_attn.parameters()),
    lr=1e-4
)

In [ ]:
from torchinfo import summary
summary(vqa_model)

In [ ]:
def train_model(epochs=3):
    vqa_model.train()

    for epoch in range(epochs):
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

        for batch in pbar:
            pixel_values = batch["pixel_values"].to(device)
            clip_input_ids = batch["clip_input_ids"].to(device)
            gpt_input_ids = batch["gpt_input_ids"].to(device)
            labels = batch["labels"].to(device)

            outputs = vqa_model(pixel_values, clip_input_ids, gpt_input_ids, labels)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    print("Training Complete!")

In [ ]:
train_model()

In [ ]:
def generate_answer(model, image_path):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    plt.imshow(image)
    plt.axis('off')
    plt.title("Query Image")
    plt.show()

    pixel_values = clip_processor(images=image, return_tensors="pt")["pixel_values"].to(device)
    question = "What breed of dog is this?"
    clip_ids = clip_tokenizer(question, return_tensors="pt")["input_ids"].to(device)
    gpt_ids = gpt_tokenizer(question, return_tensors="pt")["input_ids"].to(device)

    with torch.no_grad():
        img_feats = model.clip.vision_model(pixel_values=pixel_values).last_hidden_state
        txt_feats = model.clip.text_model(input_ids=clip_ids).last_hidden_state

        img_feats = model.image_proj(img_feats)
        txt_feats = model.text_proj(txt_feats)
        fused_feats = model.cross_attn(txt_feats, img_feats)

        question_embeds = model.decoder.transformer.wte(gpt_ids)
        current_embeds = torch.cat([fused_feats, question_embeds], dim=1)

        generated_ids = []
        print(f"Question: {question}")
        print("Answer: ", end="")

        for _ in range(15):
            outputs = model.decoder(inputs_embeds=current_embeds)
            next_token_logits = outputs.logits[:, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1)

            if next_token_id.item() == gpt_tokenizer.eos_token_id:
                break

            word = gpt_tokenizer.decode([next_token_id.item()])
            print(word, end="", flush=True)

            next_embed = model.decoder.transformer.wte(next_token_id.unsqueeze(0))
            current_embeds = torch.cat([current_embeds, next_embed], dim=1)
        print("\n")

In [ ]:
print("\nTesting on a random image from validation...")
random_row = pd.read_csv(csv_path).sample(1).iloc[0]
img_full_path = os.path.join("Images", random_row["image_name"])
generate_answer(vqa_model, img_full_path)